EECE 5644 Mini Project 2 - Skylar Denno

In [257]:
############################################################

# EECE 5644 - Machine Learning
# Skylar Denno (No group currently - all work in this project is mine)
# July 2, 2026

# MINI PROJECT 2
# Data cleaning and data preprocessing

############################################################

In [258]:
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
np.random.seed(0)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [259]:
# 1. LOAD AND INSPECT THE DATA

df = pd.read_csv("telco.csv", encoding="ISO-8859-1")
print("\nLoaded dataset with", df.shape[0], "rows\n\n")
display(df.describe())
#print(df.head(5))
print("\n\nNum. missing values in each column:\n", df.isnull().sum(), "n\n")

print(df["PhoneService"].value_counts())
# Yes, column PhoneService contains 0s and 1s only, with no missing values.



Loaded dataset with 7043 rows




,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,target,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
count,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00
mean,0.50,0.16,0.48,0.30,32.37,0.90,0.42,0.29,0.34,0.34,0.29,0.38,0.39,0.59,64.76,0.27,0.44,0.22,0.21,0.24,0.22,0.34,0.23
std,0.50,0.37,0.50,0.46,24.56,0.30,0.49,0.45,0.48,0.48,0.45,0.49,0.49,0.49,30.09,0.44,0.50,0.41,0.41,0.43,0.41,0.47,0.42
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,18.25,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,9.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,35.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,1.00,0.00,0.00,0.00,29.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,70.35,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,1.00,0.00,1.00,1.00,55.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,89.85,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00
max,1.00,1.00,1.00,1.00,72.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,118.75,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00




Num. missing values in each column:
 gender                                   0
SeniorCitizen                            0
Partner                                  0
Dependents                               0
tenure                                   0
PhoneService                             0
MultipleLines                            0
OnlineSecurity                           0
OnlineBackup                             0
DeviceProtection                         0
TechSupport                              0
StreamingTV                              0
StreamingMovies                          0
PaperlessBilling                         0
target                                   0
TotalCharges                             0
Churn                                    0
InternetService_Fiber optic              0
InternetService_No                       0
Contract_One year                        0
Contract_Two year                        0
PaymentMethod_Credit card (automatic)    0
PaymentMethod_E

In [260]:
# Some basic data cleaning:
df = df.rename(columns={
    "target" : "monthly_charge",
    "SeniorCitizen" : "senior_citizen",
    "Partner" : "partner",
    "Dependents" : "dependents",
    "PhoneService" : "phone_service",
    "MultipleLines" : "multiple_lines",
    "OnlineSecurity" : "online_security",
    "OnlineBackup" : "online_backup",
    "DeviceProtection" : "device_protection",
    "TechSupport" : "tech_support",
    "StreamingMovies" : "streaming_movies",
    "PaperlessBilling" : "paperless_billing",
    "MultipleLines" : "multiple_lines",
    "TotalCharges" : "total_charges",
    "Churn" : "churn",
    "InternetService_Fiber optic" : "fiber_optic",
    "InternetService_No" : "no_internet_service",
    "StreamingTV" : "streaming_tv"
})

# consolidating the 3 redundant payment method columns into a single one

df["PaymentMethod_Electronic check"] = df["PaymentMethod_Electronic check"].map(lambda x: x * 2)
df["PaymentMethod_Mailed check"] = df["PaymentMethod_Mailed check"].map(lambda x: x * 3)
df["payment_method_ID"] = df["PaymentMethod_Credit card (automatic)"] + df["PaymentMethod_Electronic check"] + df["PaymentMethod_Mailed check"]
df = df.drop(columns=["PaymentMethod_Credit card (automatic)", "PaymentMethod_Electronic check", "PaymentMethod_Mailed check"])
df["payment_method_name"] = df["payment_method_ID"].map({0 : "Credit Card (Manual)", 1 : "Credit Card (Automatic)", 2 : "Electronic Check", 3 : "Mailed Check"})

df["Contract_Two year"] = df["Contract_Two year"].map(lambda x: x * 2)
df["contract_length_years"] = df["Contract_One year"] + df["Contract_Two year"]
df = df.drop(columns=["Contract_One year", "Contract_Two year"])

df["gender"] = df["gender"].map({0: "Female", 1: "Male"})

# invert no_internet_service
df["internet_service"] = df["no_internet_service"].map({0: 1, 1: 0})
df = df.drop(columns=["no_internet_service"])



In [261]:
# 2. DEFINE THE VARIABLES
# feature columns, target = monthly_charge
feature_cols = ["phone_service", "multiple_lines", "online_security", "online_backup",
                "device_protection", "tech_support", "streaming_tv", "streaming_movies",
                "fiber_optic", "internet_service"]

X = df[feature_cols]
y = df["monthly_charge"]

In [262]:
# 3. SPLIT THE DATA
# form training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777)


In [263]:
# 4. FIT A MULTIVARIABLE LINEAR REGRESSION
telcomodel = LinearRegression()
telcomodel.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [264]:
# 5. REPORT THE PARAMETERS
omega_0 = telcomodel.intercept_ #intercept 
print("Learned intercept omega_0:", round(omega_0, 4))

coefficents = pd.Series(telcomodel.coef_, index=feature_cols, name="omega_i") # 10 coefficients
print("\nLearned coefficients omega_i:")
display(coefficents.to_frame())

Learned intercept omega_0: -0.1119

Learned coefficients omega_i:


,omega_i
phone_service,20.07
multiple_lines,5.00
online_security,5.02
online_backup,5.01
device_protection,5.00
tech_support,5.05
streaming_tv,9.98
streaming_movies,9.97
fiber_optic,24.95
internet_service,25.05


In [265]:
# 6. EVALUATE OUR MODEL ON THE TEST SET
y_pred = telcomodel.predict(X_test)

# Test results
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2  :", r2)
print("MAE : $^2", mae)
print("RMSE: $", rmse)

results = pd.DataFrame()
results["actual"] = y_test.values
results["predicted"] = y_pred.round(2)
results["error"] = results["actual"] - results["predicted"]
results["error_sq"] = (results["actual"] - results["predicted"]) ** 2
print(results.head(10))

# These results show that our multivariable prediction is very accurate
# and that >99% of the variance in monthly_charges can be explained and predicted by
# these 10 features.

R2  : 0.9988533669069787
MAE : $^2 0.7757835407720785
RMSE: $ 1.029169727521828
   actual  predicted  error  error_sq
0   20.05      19.96   0.09      0.01
1   58.65      59.97  -1.32      1.74
2   21.05      19.96   1.09      1.19
3   58.20      60.08  -1.88      3.53
4  101.10      99.91   1.19      1.42
5   54.35      55.08  -0.73      0.53
6   88.75      89.94  -1.19      1.42
7   79.15      79.94  -0.79      0.62
8   55.70      55.00   0.70      0.49
9   74.65      75.01  -0.36      0.13


In [266]:
# 7. COMPARE TO A SINGLE-VARIABLE BASELINE
# I will use phone_service feature as the single variable for the baseline model

X_single = df[["phone_service"]]
y = df["monthly_charge"]

X_train_single, X_test_single, y_train_single, y_test_single = train_test_split(
    X_single, y, test_size=0.2, random_state=777)

single = LinearRegression()
single.fit(X_train_single, y_train_single)

omega_0_single = single.intercept_ #intercept 
print("Learned intercept omega_0:", round(omega_0_single, 4))

coefficents_single = pd.Series(single.coef_, index=["phone_service"], name="omega_i")
print("\nLearned phone_service coefficient:")
display(coefficents_single.to_frame())

y_pred_single = single.predict(X_test_single)

# Test results
r2 = r2_score(y_test_single, y_pred_single)
mae = mean_absolute_error(y_test_single, y_pred_single)
rmse = np.sqrt(mean_squared_error(y_test_single, y_pred_single))

print("R2  :", r2)
print("MAE : $^2", mae)
print("RMSE: $", rmse)

results = pd.DataFrame()
results["actual"] = y_test_single.values
results["predicted"] = y_pred_single.round(2)
results["error"] = results["actual"] - results["predicted"]
results["error_sq"] = (results["actual"] - results["predicted"]) ** 2
print(results.head(10))

# I tried all the other feature columns individually, and fiber_optic (which is literally just an indicator 0/1)
# gave the best at an R2 of 0.08. Compared to the multivariable model, this model is terrible.
# Such a low R2 score indicates that any one single variable taken at a time is not sufficient
# in explaining all the variance in the monthly charges. The target is too reliant on multiple features.

Learned intercept omega_0: 41.8114

Learned phone_service coefficient:


,omega_i
phone_service,25.44


R2  : 0.051048221719311004
MAE : $^2 24.959528387657485
RMSE: $ 29.60716604018964
   actual  predicted  error  error_sq
0   20.05      67.25 -47.20   2227.84
1   58.65      41.81  16.84    283.59
2   21.05      67.25 -46.20   2134.44
3   58.20      67.25  -9.05     81.90
4  101.10      67.25  33.85   1145.82
5   54.35      67.25 -12.90    166.41
6   88.75      67.25  21.50    462.25
7   79.15      67.25  11.90    141.61
8   55.70      67.25 -11.55    133.40
9   74.65      67.25   7.40     54.76


In [267]:
# 8. INTERPRET THE RESULTS:

# From the multivariable model:
omega_0 = telcomodel.intercept_ #intercept 
print("Learned intercept omega_0:", round(omega_0, 4))

print("omega_0 is the model's guess of the monthly charge when all features are zero, including base (phone_service).")
print("This is basically the non-customer, no services case (should be 0).")

coefficents = pd.Series(telcomodel.coef_, index=feature_cols, name="omega_i") # 10 coefficients
print("\nLearned coefficients omega_i:")
display(coefficents.to_frame())

print("For every unit increase in each of these features, the above values are the")
print("expected change in the monthly charge for their plan.")


Learned intercept omega_0: -0.1119
omega_0 is the model's guess of the monthly charge when all features are zero, including base (phone_service).
This is basically the non-customer, no services case (should be 0).

Learned coefficients omega_i:


,omega_i
phone_service,20.07
multiple_lines,5.00
online_security,5.02
online_backup,5.01
device_protection,5.00
tech_support,5.05
streaming_tv,9.98
streaming_movies,9.97
fiber_optic,24.95
internet_service,25.05


For every unit increase in each of these features, the above values are the
expected change in the monthly charge for their plan.


In [268]:
# 9. MONTHLY CHARGE PREDICTOR - add a new user to dataframe

# ENTER A HYPOTHETICAL NEW USER'S ADD-ONS (1 = yes, 0 = no)
new_customer = {
    "phone_service": 1,
    "multiple_lines": 0,
    "online_security": 0,
    "online_backup": 1,
    "device_protection": 0,
    "tech_support": 0,
    "streaming_tv": 1,
    "streaming_movies": 1,
    "fiber_optic": 1,
    "internet_service": 1
}

y_hat = telcomodel.predict(pd.DataFrame([new_customer], columns=feature_cols))[0]
print("This user's predicted monthly charge is: $", round(y_hat, 2))

# y_hat = omega_0 + sum_i(omega_i * x_i)
# Essentially just dot-multiply the feature vector / add-on indicators with the multivariable model's coefficients
# and add the intercept to get the predicted monthly charge, as our model would predict.

This user's predicted monthly charge is: $ 94.93
